In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'MIN': ['Naz Reid']}

Out Players:
{'MIN': ['Anthony Edwards'], 'DEN': ['Aaron Gordon', 'Peyton Watson']}
Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 2 teams with confirmed lineups
Updated 1 teams with questionable players


### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
p26 = pd.read_csv('data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s26, p26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,name
69,69,2025-26,1629162,Jordan McLaughlin,Jordan,1610612759,SAS,San Antonio Spurs,42500154,2026-04-26T00:00:00,SAS @ POR,W,1.516667,1,1,1.000,0,0,0.00,0,0,0.0,0,0,0,1,0,1,0,0,1,0,2,2,6.5,0,0,5.0,1,1:31,1,166.7,166.7,166.7,104.2,75.0,75.0,62.5,91.7,91.7,1.000,0.00,50.0,0.000,0.000,0.000,0.0,0.0,1.000,1.000,0.333,0.333,93.05,110.77,92.31,110.77,0.429,3,1.0,1.0,NaN,4.38,0.13,0,0,0,5,0,0,4,0,0,0.000,1,1,1.0,0,0,0.000,43,87,0.494,14,33,0.424,14,17,0.824,9,31,40,26,13.0,12,10,5,21,20,114,21.0,115.8,116.3,92.0,93.0,23.8,23.3,0.605,2.00,19.4,0.217,0.725,0.485,0.133,0.575,0.603,99.8,99.0,82.50,98,0.646,1610612757,POR,Portland Trail Blazers,32,80,0.400,10,31,0.323,19,23,0.826,7,32,39,14,18.0,6,5,10,20,21,93,-21.0,92.0,93.0,115.8,116.3,-23.8,-23.3,0.438,0.78,10.9,0.275,0.783,0.515,0.180,0.463,0.516,99.8,99.0,82.50,100,0.354,NaN,PG,29.0,Jordan McLaughlin
70,70,2025-26,1627827,Dorian Finney-Smith,Dorian,1610612745,HOU,Houston Rockets,42500174,2026-04-26T00:00:00,HOU vs. LAL,W,19.283333,0,2,0.000,0,1,0.00,2,2,1.0,0,1,1,0,0,0,1,1,3,1,2,-13,6.2,0,0,5.0,1,19:17,1,103.4,97.2,97.2,120.1,133.3,133.3,-16.7,-36.1,-36.1,0.000,0.00,0.0,0.000,0.059,0.029,0.0,0.0,0.000,0.347,0.075,0.078,91.85,89.61,74.68,89.61,-0.020,36,0.0,2.0,NaN,3.82,1.32,3,2,5,11,0,0,9,0,1,0.000,0,1,0.0,2,3,0.667,39,81,0.481,12,30,0.400,25,31,0.806,11,24,35,19,13.0,17,4,7,22,20,115,19.0,119.0,122.3,98.7,103.2,20.3,19.1,0.487,1.46,15.0,0.333,0.625,0.471,0.138,0.556,0.608,96.9,93.5,77.92,94,0.569,1610612747,LAL,Los Angeles Lakers,37,74,0.500,5,22,0.227,17,21,0.810,10,27,37,23,24.0,6,7,4,20,22,96,-19.0,98.7,103.2,119.0,122.3,-20.3,-19.1,0.622,0.96,17.7,0.375,0.667,0.529,0.258,0.534,0.577,96.9,93.5,77.92,93,0.431,NaN,PF,32.0,Dorian Finney-Smith
72,72,2025-26,1631200,Kris Murray,Kris,1610612757,POR,Portland Trail Blazers,42500154,2026-04-26T00:00:00,POR vs. SAS,L,12.501667,2,3,0.667,0,1,0.00,0,0,0.0,0,1,1,0,0,0,0,0,1,0,4,-9,5.2,0,0,5.0,1,12:30,1,93.3,84.0,84.0,111.6,115.4,115.4,-18.4,-31.4,-31.4,0.000,0.00,0.0,0.000,0.091,0.053,0.0,0.0,0.667,0.667,0.115,0.133,94.84,97.91,81.59,97.91,0.067,25,2.0,3.0,NaN,4.41,1.01,1,4,5,11,0,0,7,2,2,1.000,0,1,0.0,1,2,0.500,32,80,0.400,10,31,0.323,19,23,0.826,7,32,39,14,18.0,6,5,10,20,21,93,-21.0,92.0,93.0,115.8,116.3,-23.8,-23.3,0.438,0.78,10.9,0.275,0.783,0.

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260427_161831.json


,home_team,away_team,commence_time,bookmakers
0,Orlando Magic,Detroit Pistons,2026-04-28 00:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
1,Phoenix Suns,Oklahoma City Thunder,2026-04-28 01:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,Denver Nuggets,Minnesota Timberwolves,2026-04-28 02:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
3,Boston Celtics,Philadelphia 76ers,2026-04-28 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,New York Knicks,Atlanta Hawks,2026-04-29 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')
pts_ast_df = pd.read_csv('data/processed/training/S26_TRAINING_PAPM.csv')

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]
print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")

lines_dfs_pts.head()

DFS latest pull: 2026-04-27 16:18:02
US latest pull: 2026-04-27 16:18:32


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Cade Cunningham,Over,28.5,-137,2026-04-28,2026-04-27T23:17:17Z,2026-04-27 16:18:02
1,PrizePicks,player_points,Cade Cunningham,Under,28.5,-137,2026-04-28,2026-04-27T23:17:17Z,2026-04-27 16:18:02
2,PrizePicks,player_points,Paolo Banchero,Over,21.5,-137,2026-04-28,2026-04-27T23:17:17Z,2026-04-27 16:18:02
3,PrizePicks,player_points,Paolo Banchero,Under,21.5,-137,2026-04-28,2026-04-27T23:17:17Z,2026-04-27 16:18:02
4,PrizePicks,player_points,Desmond Bane,Over,19.0,-137,2026-04-28,2026-04-27T23:17:17Z,2026-04-27 16:18:02


In [6]:
from difflib import get_close_matches

base_names = set(base_df['PLAYER_NAME'].unique())
missing = [n for n in pts_names if n not in base_names]

for name in missing:
    close = get_close_matches(name, base_names, n=3, cutoff=0.6)
    print(f"{name!r:35s} → {close}")

'Wendell Carter Jr'                 → ['Wendell Carter Jr.', 'Wendell Moore Jr.', 'Jevon Carter']
'Kelly Oubre Jr'                    → ['Kelly Oubre Jr.']
'R.J. Barrett'                      → ['RJ Barrett']


### Load my models

In [7]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb_2026-01-02.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb_2025-12-31.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb_2025-12-31.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb_2025-12-31.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [8]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
pts_preds.head(10)

[SKIP] Wendell Carter Jr: min_pipeline returned None (need >= 10 games)
[SKIP] Kelly Oubre Jr: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)
[SKIP] Wendell Carter Jr: min_pipeline returned None (need >= 10 games)
[SKIP] Kelly Oubre Jr: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,Cade Cunningham,PTS,29.21,37.78,42.31,0.4079,0.6522,0.9246,11.91,24.64,39.12,"[0.7754010695187166, 0.2580645161290322, 0.660..."
1,Paolo Banchero,PTS,29.29,38.11,41.49,0.3987,0.6156,0.8766,11.68,23.46,36.37,"[1.0617059891107077, 0.9007506255212676, 0.796..."
2,Desmond Bane,PTS,25.05,34.17,39.90,0.2828,0.5049,0.7774,7.09,17.25,31.01,"[0.5233453052847614, 0.467032967032967, 0.6616..."
3,Franz Wagner,PTS,22.54,31.28,37.21,0.4109,0.6416,0.8922,9.26,20.07,33.20,"[0.9393346379647748, 0.9610983981693364, 0.674..."
4,Tobias Harris,PTS,23.46,33.52,39.83,0.2495,0.4882,0.7271,5.85,16.36,28.96,"[0.3186646433990895, 0.5172413793103449, 0.438..."
5,Jalen Duren,PTS,19.68,26.37,33.24,0.2462,0.4953,0.7261,4.85,13.06,24.13,"[0.968392737054472, 1.07326178254783, 0.6, 0.6..."
6,Jalen Suggs,PTS,23.98,32.96,39.67,0.2379,0.5185,0.8053,5.70,17.09,31.95,"[0.4109589041095891, 0.4081632653061224, 0.235..."
7,Duncan Robinson,PTS,20.03,28.59,35.62,0.1495,0.3727,0.6320,2.99,10.66,22.51,"[0.3041825095057034, 0.5191594561186651, 0.378..."
8,Ausar Thompson,PTS,21.36,29.86,38.30,0.1484,0.3655,0.6074,3.17,10.91,23.26,"[0.1674730182359508, 0.4855460144764645, 0.379..."
9,Anthony Black,PTS,17.33,23.00,29.45,0.1936,0.4319,0.6930,3.35,9.93,20.41,"[0.5257836198179979, 0.6417112299465241, 0.222..."


### Get Line Probabilities

In [10]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, run_pts_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, run_pts_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
31,Mikal Bridges,AST,2.0,21.33,30.52,37.44,0.03,2.09,5.45,0.482,0.518
179,Joel Embiid,PTS,26.5,23.18,28.75,35.24,13.45,23.00,36.91,0.448,0.552
117,Marcus Smart,REB,2.5,25.87,35.91,40.61,1.01,3.78,7.78,0.627,0.373
85,Joel Embiid,REB,8.0,23.18,28.75,35.24,3.83,7.89,15.20,0.506,0.494
83,Tim Hardaway Jr,REB,2.5,19.29,25.91,32.66,0.63,2.76,6.58,0.671,0.329
136,Desmond Bane,PTS,18.5,25.05,34.17,39.90,7.09,17.25,31.01,0.497,0.503
41,Reed Sheppard,AST,4.5,21.44,29.93,41.86,1.19,4.35,9.91,0.559,0.441
194,Karl-Anthony Towns,PTS,20.5,25.17,33.71,39.16,8.90,19.44,33.32,0.583,0.417
106,Jrue Holiday,REB,5.0,28.53,37.02,42.60,1.73,4.96,9.39,0.551,0.449
189,Nikola Vucevic,PTS,6.5,17.45,22.77,30.68,3.46,9.56,20.61,0.739,0.261


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
194,Karl-Anthony Towns,PTS,20.5,25.17,33.71,39.16,8.90,19.44,33.32,0.583,0.417,PTS,Underdog,Atlanta Hawks,-6.5,213.8,112.9,10.0,102.50,5.0,-105.0,-113.0,0.512,0.531,20.0,20.5,2.98,-0.5,0.0,0.168,0.433,0.567,-15.46,6.88,0.6,0.5,0.60,0.48,31.98,2.05,0.24,0.05,23.50,6.0
232,LeBron James,PTS,23.5,31.01,38.54,44.08,12.35,25.21,39.86,0.601,0.399,PTS,Underdog,Houston Rockets,-2.5,207.5,112.1,6.0,96.98,29.0,-137.0,-137.0,0.578,0.578,21.5,22.5,7.55,-2.0,-1.0,0.265,0.396,0.604,-31.49,4.49,0.4,0.5,0.33,0.36,33.17,7.98,0.30,0.08,21.71,7.0
193,Jalen Johnson,PTS,20.5,25.15,36.00,42.68,6.94,18.58,33.38,0.438,0.562,PTS,Underdog,New York Knicks,6.5,213.8,112.3,7.0,97.71,25.0,-105.0,-107.0,0.512,0.517,18.5,18.0,3.72,-2.0,-2.5,0.538,0.295,0.705,-42.40,36.39,0.4,0.3,0.40,0.59,34.45,4.88,0.25,0.03,19.57,7.0
167,Jamal Murray,PTS,27.5,30.62,39.68,43.68,10.95,23.41,38.19,0.397,0.603,PTS,Underdog,Minnesota Timberwolves,-11.5,224.0,112.5,8.0,101.50,10.0,-105.0,-112.0,0.512,0.528,25.5,28.0,7.34,-2.0,0.5,0.272,0.393,0.607,-23.27,14.90,0.6,0.5,0.47,0.38,38.88,3.77,0.25,0.05,29.00,8.0
117,Marcus Smart,REB,2.5,25.87,35.91,40.61,1.01,3.78,7.78,0.627,0.373,REB,Underdog,Houston Rockets,-2.5,207.5,112.1,6.0,96.98,29.0,-137.0,-137.0,0.578,0.578,2.3,2.0,1.42,-0.2,-0.5,0.141,0.444,0.556,-23.19,-3.82,0.2,0.3,0.40,0.55,31.15,6.06,0.17,0.07,2.14,7.0


In [12]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
33,Stephon Castle,AST,7.5,25.78,35.41,40.92,2.83,6.62,12.58,0.524,0.476,AST,PrizePicks,Portland Trail Blazers,-12.0,215.2,113.5,12.0,101.63,9.0,-137.0,-137.0,0.578,0.578,8.2,8.5,2.74,0.7,1.0,-0.255,0.601,0.399,3.97,-30.98,0.4,0.6,0.67,0.47,33.19,4.00,0.24,0.04,6.80,5.0
139,Jalen Duren,PTS,15.5,19.68,26.37,33.24,4.85,13.06,24.13,0.419,0.581,PTS,PrizePicks,Orlando Magic,-3.5,214.5,113.6,13.0,100.56,14.0,100.0,-107.0,0.500,0.517,16.5,17.0,7.40,1.0,1.5,-0.135,0.554,0.446,10.80,-13.72,0.4,0.6,0.73,0.71,30.25,4.00,0.20,0.05,14.00,7.0
164,Royce O'Neale,PTS,5.5,18.60,25.40,31.86,1.16,6.78,16.72,0.609,0.391,PTS,PrizePicks,Oklahoma City Thunder,11.5,215.5,106.5,1.0,100.37,16.0,-105.0,105.0,0.512,0.488,8.0,6.5,5.40,2.5,1.0,-0.463,0.678,0.322,32.37,-33.99,0.6,0.5,0.67,0.80,23.52,5.40,0.11,0.05,7.57,7.0
69,Collin Gillespie,REB,3.5,20.65,30.60,37.86,1.20,4.35,9.31,0.699,0.301,REB,PrizePicks,Oklahoma City Thunder,11.5,215.5,106.5,1.0,100.37,16.0,-116.0,105.0,0.537,0.488,4.4,4.0,2.76,0.9,0.5,-0.326,0.628,0.372,16.94,-23.74,0.8,0.5,0.47,0.55,25.92,6.24,0.16,0.03,4.71,7.0
194,Karl-Anthony Towns,PTS,20.5,25.17,33.71,39.16,8.90,19.44,33.32,0.583,0.417,PTS,PrizePicks,Atlanta Hawks,-6.5,213.8,112.9,10.0,102.50,5.0,-105.0,-113.0,0.512,0.531,20.0,20.5,2.98,-0.5,0.0,0.168,0.433,0.567,-15.46,6.88,0.6,0.5,0.60,0.48,31.98,2.05,0.24,0.05,23.50,6.0


In [13]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
186,VJ Edgecombe,PTS,12.5,31.27,39.54,43.93,7.22,19.71,33.54,0.725,0.275,PTS,Betr DFS,Boston Celtics,11.5,214.0,111.7,4.0,95.58,30.0,102.0,-125.0,0.495,0.556,14.5,13.5,7.15,2.0,1.0,-0.280,0.610,0.390,23.22,-29.80,0.4,0.6,0.73,0.62,37.08,3.31,0.20,0.05,17.38,8.0
110,Keldon Johnson,REB,3.5,17.61,23.54,30.16,1.87,5.03,10.50,0.844,0.156,REB,Betr DFS,Portland Trail Blazers,-12.0,215.2,113.5,12.0,101.63,9.0,-137.0,-137.0,0.578,0.578,5.4,6.0,2.50,1.9,2.5,-0.760,0.776,0.224,34.24,-61.25,0.8,0.8,0.67,0.72,22.00,4.09,0.22,0.06,6.14,7.0
231,Jamal Shead,PTS,6.5,20.64,30.07,37.97,2.23,8.68,20.70,0.583,0.417,PTS,Betr DFS,Cleveland Cavaliers,9.0,216.0,114.1,15.0,100.70,13.0,-137.0,-137.0,0.578,0.578,6.4,5.5,4.72,-0.1,-1.0,0.021,0.492,0.508,-14.89,-12.12,0.2,0.4,0.47,0.45,26.30,4.94,0.14,0.05,7.14,7.0
160,Cason Wallace,PTS,7.5,18.27,24.45,30.88,1.91,7.24,16.59,0.433,0.567,PTS,Betr DFS,Phoenix Suns,-11.5,215.5,112.9,9.0,98.14,24.0,102.0,-112.0,0.495,0.528,7.9,6.0,6.23,0.4,-1.5,-0.064,0.526,0.474,6.25,-10.28,0.0,0.4,0.33,0.46,22.71,4.22,0.13,0.04,6.00,7.0
73,Rudy Gobert,REB,11.5,25.74,34.44,39.59,5.95,12.18,20.79,0.525,0.475,REB,Betr DFS,Denver Nuggets,11.5,224.0,116.0,21.0,99.49,20.0,-137.0,-137.0,0.578,0.578,11.1,11.0,2.96,0.1,0.0,-0.034,0.514,0.486,-11.08,-15.93,0.6,0.5,0.67,0.57,31.74,3.85,0.13,0.03,10.25,8.0


In [14]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
150,Shai Gilgeous-Alexander,PTS,31.5,29.15,38.25,41.84,16.58,31.08,44.72,0.609,0.391,PTS,DraftKings Pick6,Phoenix Suns,-11.5,215.5,112.9,9.0,98.14,24.0,-110.0,-106.0,0.524,0.515,29.9,26.5,9.19,-1.6,-5.0,0.174,0.431,0.569,-17.72,10.58,0.4,0.3,0.40,0.42,32.39,5.55,0.34,0.04,32.33,6.0
175,Aaron Gordon,PTS,11.5,19.79,26.66,33.19,3.83,11.10,22.01,0.518,0.482,PTS,DraftKings Pick6,Minnesota Timberwolves,-11.5,224.0,112.5,8.0,101.50,10.0,105.0,-135.0,0.488,0.574,12.9,13.5,5.63,1.4,2.0,-0.249,0.598,0.402,22.59,-30.02,0.4,0.6,0.67,0.74,29.74,5.76,0.18,0.05,13.20,5.0
170,Jaden McDaniels,PTS,17.5,26.37,35.62,42.04,5.78,16.73,30.60,0.491,0.509,PTS,DraftKings Pick6,Denver Nuggets,11.5,224.0,116.0,21.0,99.49,20.0,100.0,-117.0,0.500,0.539,17.3,16.5,3.56,-0.2,-1.0,0.056,0.478,0.522,-4.40,-3.18,0.2,0.4,0.27,0.34,33.47,6.49,0.22,0.07,17.88,8.0
16,Julius Randle,AST,4.5,28.74,38.09,42.19,1.32,4.02,8.33,0.569,0.431,AST,DraftKings Pick6,Denver Nuggets,11.5,224.0,116.0,21.0,99.49,20.0,-120.0,110.0,0.545,0.476,3.9,4.0,1.37,-0.6,-0.5,0.438,0.331,0.669,-39.32,40.49,0.2,0.2,0.27,0.47,32.60,2.71,0.26,0.04,4.75,8.0
158,Collin Gillespie,PTS,7.5,20.65,30.60,37.86,2.26,11.87,25.23,0.694,0.306,PTS,DraftKings Pick6,Oklahoma City Thunder,11.5,215.5,106.5,1.0,100.37,16.0,-118.0,114.0,0.541,0.467,7.7,7.5,3.68,0.2,0.0,-0.054,0.522,0.478,-3.56,2.29,0.4,0.5,0.60,0.75,25.92,6.24,0.16,0.03,8.86,7.0


In [15]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
76,Jamal Murray,REB,4.5,30.62,39.68,43.68,1.32,4.67,9.19,0.566,0.434,REB,PrizePicks,Minnesota Timberwolves,-11.5,224.0,112.5,8.0,101.50,10.0,-137.0,-137.0,0.578,0.578,4.9,5.0,1.85,-0.1,0.0,0.054,0.478,0.522,-17.31,-9.70,0.4,0.4,0.40,0.37,38.88,3.77,0.25,0.05,4.88,8.0
171,Naz Reid,PTS,12.5,16.70,21.59,28.74,3.76,10.37,21.32,0.312,0.688,PTS,DraftKings Pick6,Denver Nuggets,11.5,224.0,116.0,21.0,99.49,20.0,-115.0,105.0,0.535,0.488,11.9,12.0,5.20,-0.6,-0.5,0.115,0.454,0.546,-15.12,11.93,0.4,0.4,0.33,0.52,24.63,4.36,0.21,0.03,12.38,8.0
23,Payton Pritchard,AST,4.0,20.67,28.46,36.25,1.06,3.68,8.45,0.529,0.471,AST,Betr DFS,Philadelphia 76ers,-11.5,214.0,114.4,17.0,100.40,15.0,-137.0,-137.0,0.578,0.578,5.0,5.0,2.26,1.0,1.0,-0.442,0.671,0.329,16.08,-43.09,0.8,0.6,0.47,0.59,31.16,3.22,0.20,0.05,4.12,8.0
160,Cason Wallace,PTS,7.5,18.27,24.45,30.88,1.91,7.24,16.59,0.433,0.567,PTS,PrizePicks,Phoenix Suns,-11.5,215.5,112.9,9.0,98.14,24.0,102.0,-112.0,0.495,0.528,7.9,6.0,6.23,0.4,-1.5,-0.064,0.526,0.474,6.25,-10.28,0.0,0.4,0.33,0.46,22.71,4.22,0.13,0.04,6.00,7.0
55,Franz Wagner,REB,5.0,22.54,31.28,37.21,1.33,4.40,8.61,0.465,0.535,REB,PrizePicks,Detroit Pistons,3.5,214.5,108.9,2.0,99.88,19.0,-137.0,-137.0,0.578,0.578,3.3,4.0,2.41,-1.7,-1.0,0.705,0.240,0.760,-58.48,31.47,0.2,0.1,0.13,0.46,24.61,5.94,0.26,0.06,6.20,5.0


### Get top EVs for 2 legs

In [16]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 163  |  Pairs: 537  |  Slate: 9  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 57  |  Pairs: 94  |  Slate: 4  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [18]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 52  |  Pairs: 25  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json


In [19]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 144  |  Pairs: 462  |  Slate: 8  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json


### Top EVs for 3 Legs

In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 163  |  Triples: 16024  |  Slate: 7  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json


In [21]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 57  |  Triples: 1147  |  Slate: 3  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json


In [22]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 144  |  Triples: 12289  |  Slate: 7  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json


In [23]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 52  |  Triples: 156  |  Slate: 1  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
